In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

#PATHS
RAW = Path("../data/raw")
PROCESSED = Path("../data/processed")
PROCESSED.mkdir(exist_ok=True)

In [2]:
# IDENTIFYING SCHEMA AND MISSING DATA
txn=pd.read_csv(RAW / "transactions.csv")
print(txn.shape)   #rows and columns
print(txn.dtypes)  #column data types
print(txn.head())  #first 5 rows
print(txn.isnull().sum())  #null count per column

(53704, 7)
Date                    object
SKU_ID                  object
Category                object
Listed_Price_INR       float64
Discount_Pct           float64
Effective_Price_INR    float64
Quantity_Sold            int64
dtype: object
         Date   SKU_ID     Category  Listed_Price_INR  Discount_Pct  \
0  2022-01-01  SKU0001  Electronics           9513.37           0.0   
1  2022-01-01  SKU0002  Electronics          20969.08           0.0   
2  2022-01-01  SKU0003  Electronics          36809.42           0.0   
3  2022-01-01  SKU0004  Electronics          11274.10           0.0   
4  2022-01-01  SKU0005  Electronics           6890.42           0.0   

   Effective_Price_INR  Quantity_Sold  
0              9513.37             26  
1             20969.08             18  
2             36809.42             25  
3             11274.10             17  
4              6890.42             21  
Date                   0
SKU_ID                 0
Category               0
Listed_Price_INR

In [3]:
# IDENTIFYING DUPLICATE ROWS
print("Duplicate Rows: ", txn.duplicated().sum())
print("Duplicate Txns: ", txn.duplicated(subset=["SKU_ID","Date"]).sum())

Duplicate Rows:  0
Duplicate Txns:  0


In [4]:
#STANDARDISE DATE
txn["Date"] = pd.to_datetime(txn["Date"], format='ISO8601').dt.strftime("%Y-%m-%d")
print(txn["Date"].head())
print(txn["Date"].dtype)

0    2022-01-01
1    2022-01-01
2    2022-01-01
3    2022-01-01
4    2022-01-01
Name: Date, dtype: object
object


In [5]:
# VALIDATING SELLING PRICE >= COST PRICE
master = pd.read_csv(RAW / "product_master.csv")

txn = txn.merge(master[["SKU_ID", "Base_Cost_INR"]], on="SKU_ID", how="left")

txn["price_violation"] = txn["Effective_Price_INR"] < txn["Base_Cost_INR"]
print("Price violations:", txn["price_violation"].sum())

violations = txn[txn["price_violation"]]
print(violations[["SKU_ID", "Effective_Price_INR", "Base_Cost_INR"]].head(20))

Price violations: 3417
      SKU_ID  Effective_Price_INR  Base_Cost_INR
931  SKU0001              7961.77           8000
932  SKU0002             16045.99          18000
933  SKU0003             29942.59          30000
934  SKU0004              8992.68           9000
935  SKU0005              5159.61           6000
936  SKU0006             36060.06          42000
937  SKU0007             11161.79          12000
938  SKU0008               542.45            550
941  SKU0011               471.55            520
942  SKU0012               351.99            360
943  SKU0013               253.67            280
945  SKU0015              1522.88           1600
949  SKU0019              1475.44           1500
950  SKU0020              4922.11           5000
952  SKU0022              3017.45           3200
954  SKU0024              1724.27           1800
955  SKU0025              1218.12           1400
957  SKU0027               490.28            500
960  SKU0030              2192.67           22

In [6]:
## CLEANED-TRANSACTIONS SAVED AS PARQUET
txn_clean = txn.drop_duplicates(subset=["SKU_ID", "Date"]) #0 duplicates, but defensive approach

# Ensuring the shape passed is clean. (Defensive approach)
print("Clean shape:", txn_clean.shape)
print("Price violations retained:", txn_clean["price_violation"].sum())

#Save to the processed folder
txn_clean.to_parquet(PROCESSED / "transactions_clean.parquet", index=False)
print("Saved: transactions_clean.parquet")

Clean shape: (53704, 9)
Price violations retained: 3417
Saved: transactions_clean.parquet


In [7]:
## LOAD AND PROFILE COMPTETITOR_PRICES.CSV
comp = pd.read_csv(RAW / "competitor_prices.csv")

print("Shape:", comp.shape)
print("\nColumn names:")
print(comp.columns.tolist())
print("\nData types:")
print(comp.dtypes)
print("\nFirst 5 rows:")
print(comp.head())
print("\nNull counts:")
print(comp.isnull().sum())

Shape: (23079, 4)

Column names:
['SKU_ID', 'Competitor_Name', 'Week_Start_Date', 'Competitor_Price_INR']

Data types:
SKU_ID                   object
Competitor_Name          object
Week_Start_Date          object
Competitor_Price_INR    float64
dtype: object

First 5 rows:
    SKU_ID Competitor_Name Week_Start_Date  Competitor_Price_INR
0  SKU0001        MegaMart      2022-01-03             8918.8860
1  SKU0001        QuickBuy      2022-01-03             9005.5527
2  SKU0001        ShopKart      2022-01-03             8619.8166
3  SKU0001        MegaMart      2022-01-10             8641.7367
4  SKU0001        QuickBuy      2022-01-10             9125.6529

Null counts:
SKU_ID                  0
Competitor_Name         0
Week_Start_Date         0
Competitor_Price_INR    0
dtype: int64


In [8]:
# IDENTIFYING DUPLICATE ROWS
print("Duplicate Rows: ", comp.duplicated().sum())
print("Duplicate Entries: ", comp.duplicated(subset=["SKU_ID","Competitor_Name","Week_Start_Date"]).sum())

Duplicate Rows:  0
Duplicate Entries:  0


In [9]:
## COMPARING THE PRODUCT CATALOGUE IN OUR AND THE COMPETITOR's COMPANY
comp_skus=set(comp["SKU_ID"].unique()) #competitor's catalogue
master_skus=set(master["SKU_ID"].unique()) #master is the catalogue of our company

print("Products in comp's catalogue but not in ours:",comp_skus-master_skus)
print("Products in our catalogue but not in comp's:", master_skus-comp_skus)
print("Products in both companies catalogue:", len(comp_skus & master_skus))

Products in comp's catalogue but not in ours: set()
Products in our catalogue but not in comp's: set()
Products in both companies catalogue: 49


In [10]:
## STANDARDISING THE DATES IN COMPETITOR_PRICES.CSV
comp["Week_Start_Date"]=pd.to_datetime(comp["Week_Start_Date"], format='ISO8601').dt.strftime("%Y-%m-%d")

print("Date sample after standardisation:")
print(comp["Week_Start_Date"].head())

Date sample after standardisation:
0    2022-01-03
1    2022-01-03
2    2022-01-03
3    2022-01-10
4    2022-01-10
Name: Week_Start_Date, dtype: object


In [11]:
## PROFILING PRODUCT_MASTER.CSV

print("Shape:", master.shape)
print("\nColumn names:")
print(master.columns.tolist())
print("\nData types:")
print(master.dtypes)
print("\nFirst 5 rows:")
print(master.head())
print("\nNull counts:")
print(master.isnull().sum())

Shape: (49, 6)

Column names:
['SKU_ID', 'Product_Name', 'Category', 'Base_Cost_INR', 'Margin_Target_Pct', 'Shelf_Life_Days']

Data types:
SKU_ID                object
Product_Name          object
Category              object
Base_Cost_INR          int64
Margin_Target_Pct    float64
Shelf_Life_Days      float64
dtype: object

First 5 rows:
    SKU_ID           Product_Name     Category  Base_Cost_INR  \
0  SKU0001          Smart Earbuds  Electronics           8000   
1  SKU0002    4K Smart TV 43-inch  Electronics          18000   
2  SKU0003      Gaming Console X1  Electronics          30000   
3  SKU0004    Smartwatch Series 5  Electronics           9000   
4  SKU0005  Bluetooth Speaker Pro  Electronics           6000   

   Margin_Target_Pct  Shelf_Life_Days  
0               0.18              NaN  
1               0.18              NaN  
2               0.18              NaN  
3               0.18              NaN  
4               0.18              NaN  

Null counts:
SKU_ID       

In [12]:
## CHECKING THE DATA QUALITY OF PRODUCT_MASTER.CSV
print ("Total rows in master:", len(master))
print ("Unique SKUs in master:", master["SKU_ID"].nunique())
print ("Duplicate SKU rows:", master.duplicated(subset =["SKU_ID"]).sum())

Total rows in master: 49
Unique SKUs in master: 49
Duplicate SKU rows: 0


In [13]:
## VERIFYING ALL TRANSACTION SKUS EXIST IN MASTER
txn_skus = set(txn_clean["SKU_ID"].unique())
master_skus = set(master["SKU_ID"].unique())
missing_from_master = txn_skus - master_skus
print("Transaction SKUs missing from master:", missing_from_master)
print("Count missing:", len(missing_from_master))

Transaction SKUs missing from master: set()
Count missing: 0


In [14]:
#Ensuring that the margin pct is in decimal form
print("Margin_Target_Pct sample:")
print(master["Margin_Target_Pct"].describe())

Margin_Target_Pct sample:
count    4.900000e+01
mean     1.800000e-01
std      2.804321e-17
min      1.800000e-01
25%      1.800000e-01
50%      1.800000e-01
75%      1.800000e-01
max      1.800000e-01
Name: Margin_Target_Pct, dtype: float64


In [15]:
#Rename for convenience
master=master.rename(columns={"Margin_Target_Pct":"margin_pct"})

In [16]:
#Verify Cost Price makes sense
print("\nBase_Cost_INR stats:")
print(master["Base_Cost_INR"].describe())


Base_Cost_INR stats:
count       49.000000
mean      3783.265306
std       7599.339442
min        250.000000
25%        520.000000
50%       1400.000000
75%       2800.000000
max      42000.000000
Name: Base_Cost_INR, dtype: float64


In [17]:
master.to_parquet(PROCESSED / "product_master_clean.parquet", index=False)
print("Saved: product_master_clean.parquet")
print("Columns saved:",master.columns.tolist())

Saved: product_master_clean.parquet
Columns saved: ['SKU_ID', 'Product_Name', 'Category', 'Base_Cost_INR', 'margin_pct', 'Shelf_Life_Days']
